# SQL Query Practice — WHERE, Subqueries & NULL Handling

Notes from SQL Zoo–style practice: `OR`/`XOR` logic, `LIKE` patterns, `COUNT` vs `COUNT(DISTINCT)`, `ALL`/`ANY`, correlated subqueries, `GROUP BY`/`HAVING`, and NULL comparison rules.

**How to revise:** each topic has a notes cell (question + concept) above its query cell (the SQL and alternatives). The `world` and `nobel` tables are the classic SQL Zoo practice tables — create them in your workspace to run these queries.

## 1. Basic WHERE with OR

**Question:** Get countries with a large area OR a large population.

* `OR` keeps a row when *at least one* condition is true.
* `AND` keeps a row only when *both* are true — the classic interview distinction.

In [0]:
%sql
SELECT name, population, area
FROM world
WHERE area >= 3000000
   OR population >= 25000000

## 2. XOR Logic (exactly one condition)

**Question:** Show countries meeting exactly ONE of the two conditions — not both.

* `XOR` = exclusive OR: true only when the two conditions have *different* truth values.
* PostgreSQL has the `XOR` operator; standard SQL expresses it as "the two comparisons disagree": `(cond1) != (cond2)`.

In [0]:
%sql
-- PostgreSQL
SELECT name, population, area
FROM world
WHERE area > 3000000 XOR population > 250000000;

-- Standard SQL alternative
SELECT name, population, area
FROM world
WHERE (area > 3000000) != (population > 250000000)

## 3. Multiple LIKE — all vowels, no spaces

**Question:** Find country names containing every vowel (a, e, i, o, u) and no space.

* Each `LIKE '%x%'` is a separate condition, all joined with `AND`.
* `%` matches any sequence of characters (including none).

In [0]:
%sql
SELECT name
FROM world
WHERE name LIKE '%a%'
  AND name LIKE '%e%'
  AND name LIKE '%i%'
  AND name LIKE '%o%'
  AND name LIKE '%u%'
  AND name NOT LIKE '% %'

## 4. LIKE pattern — exact word prefix

**Question:** Find Nobel winners whose first name is exactly "John" (not Johnathan, Johnny).

* `'John %'` = starts with "John" followed by a space, so longer "John..." names are excluded.

In [0]:
%sql
SELECT winner
FROM nobel
WHERE winner LIKE 'John %'

## 5. Combining AND with OR

**Question:** Show Physics winners from 1980 plus Chemistry winners from 1984.

* When mixing `AND` and `OR`, wrap each `AND` pair in parentheses — `AND` binds tighter than `OR`, so without brackets the logic silently changes.

In [0]:
%sql
SELECT yr, subject, winner
FROM nobel
WHERE (yr = 1980 AND subject = 'physics')
   OR (yr = 1984 AND subject = 'chemistry')

## 6. Unicode characters

**Question:** Find the winner 'Peter Grünberg'.

* Match the Unicode character directly when editor and table encoding support it.
* `E'...\u00FC'` is PostgreSQL escape-string syntax (not standard SQL) — prefer the direct character.

In [0]:
%sql
-- Direct match
SELECT * FROM nobel
WHERE winner = 'Peter Grünberg';

-- PostgreSQL escape-string alternative
SELECT * FROM nobel
WHERE winner = E'Peter Gr\u00FCnberg'

## 7. COUNT vs COUNT(DISTINCT ...)

**Question:** "How many years had no Medicine prize?"

* `COUNT(col)` counts *rows* — a year with 3 prizes is counted 3 times.
* `COUNT(DISTINCT col)` counts *unique values* — correct for "how many years".
* Interview takeaway: `COUNT(*)` = all rows, `COUNT(col)` = non-null rows, `COUNT(DISTINCT col)` = unique non-null values.

In [0]:
%sql
-- WRONG for "how many years": counts every prize row
SELECT COUNT(yr)
FROM nobel
WHERE yr NOT IN (SELECT DISTINCT yr FROM nobel WHERE subject = 'Medicine');

-- CORRECT: counts unique years
SELECT COUNT(DISTINCT yr)
FROM nobel
WHERE yr NOT IN (SELECT DISTINCT yr FROM nobel WHERE subject = 'Medicine')

## 8. Subquery with NOT IN

**Question:** List the years in which no Medicine prize was given.

* `NOT IN (subquery)` keeps rows whose value appears nowhere in the subquery result.
* Caveat: if the subquery returns even one NULL, `NOT IN` matches nothing — same NULL rule as topic 10.

In [0]:
%sql
SELECT DISTINCT yr
FROM nobel
WHERE yr NOT IN (SELECT DISTINCT yr FROM nobel WHERE subject = 'Medicine')

## 9. Comparison with ALL

**Question:** Which countries have a GDP bigger than EVERY European country?

* `> ALL (subquery)` = greater than the maximum of the list.
* `> ANY (subquery)` = greater than at least one value (i.e., greater than the minimum) — the standard ALL vs ANY interview question.

In [0]:
%sql
-- Using ALL
SELECT name
FROM world
WHERE gdp > ALL (SELECT gdp
                 FROM world
                 WHERE continent = 'Europe'
                   AND gdp IS NOT NULL);

-- Equivalent, usually easier to read
SELECT name
FROM world
WHERE gdp > (SELECT MAX(gdp)
             FROM world
             WHERE continent = 'Europe')

## 10. NULL breaks ALL / ANY

**Rule:** any comparison with NULL returns NULL (not TRUE/FALSE), and WHERE keeps only TRUE rows.

* If the subquery list contains a NULL, `gdp > ALL (...)` can never be TRUE → **no rows returned**.
* Fix: filter NULLs inside the subquery with `AND gdp IS NOT NULL`.

In [0]:
%sql
-- WRONG: NULL gdp in the list → returns no results
SELECT name
FROM world
WHERE gdp > ALL (SELECT gdp FROM world WHERE continent = 'Europe');

-- CORRECT: filter NULLs
SELECT name
FROM world
WHERE gdp > ALL (SELECT gdp
                 FROM world
                 WHERE continent = 'Europe'
                   AND gdp IS NOT NULL)

## 11. Max per group — correlated subquery

**Question:** Show the largest country (by area) in each continent.

* A correlated subquery references the outer query (`y.continent = x.continent`), so it is re-evaluated for each outer row.
* Same pattern works with `MIN` or `<= ALL`.

In [0]:
%sql
-- Correlated subquery with MAX
SELECT continent, name, area
FROM world x
WHERE area = (SELECT MAX(area)
              FROM world y
              WHERE y.continent = x.continent);

-- Alternative using ALL
SELECT continent, name, area
FROM world x
WHERE area >= ALL (SELECT area
                   FROM world y
                   WHERE y.continent = x.continent
                     AND area IS NOT NULL)

## 12. Min per group — alphabetically first

**Question:** Show the first country alphabetically in each continent.

* `MIN(name)` works on text too — it returns the alphabetically smallest value.

In [0]:
%sql
-- Using MIN
SELECT continent, name
FROM world x
WHERE name = (SELECT MIN(name)
              FROM world y
              WHERE y.continent = x.continent);

-- Alternative using ALL
SELECT continent, name
FROM world x
WHERE name <= ALL (SELECT name
                   FROM world y
                   WHERE y.continent = x.continent)

## 13. Filter groups with HAVING (or NOT EXISTS)

**Question:** Show countries in continents where EVERY country has population ≤ 25M.

* `WHERE` filters rows *before* grouping; `HAVING` filters *groups after* aggregation.
* Trick: `HAVING MAX(population) <= 25000000` — if the group's max is small, every member is small.
* `NOT EXISTS` says the same thing set-based: no country in that continent exceeds 25M.

In [0]:
%sql
-- Using GROUP BY + HAVING
SELECT name, continent, population
FROM world
WHERE continent IN (SELECT continent
                    FROM world
                    GROUP BY continent
                    HAVING MAX(population) <= 25000000);

-- Alternative using NOT EXISTS
SELECT name, continent, population
FROM world x
WHERE NOT EXISTS (SELECT 1
                  FROM world y
                  WHERE y.continent = x.continent
                    AND y.population > 25000000)

## 14. Exclude self when comparing within the same table

**Question:** Find countries whose population is more than 3× every other country in its continent.

* When a subquery compares rows of the *same* table, exclude the row itself: `x.name != y.name`.
* Without self-exclusion the country is compared against itself — wrong result.

In [0]:
%sql
SELECT name, continent
FROM world x
WHERE population > ALL (SELECT 3 * population
                        FROM world y
                        WHERE y.continent = x.continent
                          AND x.name != y.name
                          AND population IS NOT NULL)

## Key Concepts Summary

| Concept | Rule / Distinction |
| --- | --- |
| `OR` vs `AND` | OR = at least one true; AND = both true |
| `XOR` | Exactly one true — standard SQL: `(cond1) != (cond2)` |
| `ALL` vs `ANY` | `> ALL` = greater than every value (the max); `> ANY` = greater than at least one (the min) |
| NULL comparisons | Comparing with NULL → NULL; WHERE keeps only TRUE; add `IS NOT NULL` in ALL/ANY subqueries |
| COUNT variants | `COUNT(*)` = all rows; `COUNT(col)` = non-null rows; `COUNT(DISTINCT col)` = unique non-null values |
| Correlated subquery | Inner query references the outer row (`y.continent = x.continent`) |
| `WHERE` vs `HAVING` | WHERE filters rows before grouping; HAVING filters groups after aggregation |
| Self-exclusion | Comparing within the same table: add `AND x.name != y.name` |
| `NOT IN` + NULL | A NULL in the subquery list makes `NOT IN` return no rows — filter NULLs first |